In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error

In [2]:
brent_daily = pd.read_csv(r"datasets/brent_daily.csv")
brent_weekly = pd.read_csv(r"datasets/brent-weekly.csv")
brent_monthly = pd.read_csv(r"datasets/brent-monthly.csv")
brent_yearly = pd.read_csv(r"datasets/brent-year.csv")

wti_daily = pd.read_csv(r"datasets/wti-daily.csv")
wti_weekly = pd.read_csv(r"datasets/wti-weekly.csv")
wti_monthly = pd.read_csv(r"datasets/wti-monthly.csv")
wti_yearly = pd.read_csv(r"datasets/wti-year.csv")

In [3]:
print(brent_daily.shape)
print(brent_weekly.shape)
print(brent_monthly.shape)
print(brent_yearly.shape)

print(wti_daily.shape)
print(wti_weekly.shape)
print(wti_monthly.shape)
print(wti_yearly.shape)

(9922, 2)
(2042, 2)
(470, 2)
(39, 2)
(10191, 2)
(2113, 2)
(486, 2)
(40, 2)


In [4]:
print(brent_daily.head())
print(wti_daily.head())

         Date  Price
0  1987-05-20  18.63
1  1987-05-21  18.45
2  1987-05-22  18.55
3  1987-05-25  18.60
4  1987-05-26  18.63
         Date  Price
0  1986-01-02  25.56
1  1986-01-03  26.00
2  1986-01-06  26.53
3  1986-01-07  25.85
4  1986-01-08  25.87


In [5]:
print(brent_daily.tail())
print(wti_daily.tail())

            Date  Price
9917  2026-06-23  75.69
9918  2026-06-24  72.09
9919  2026-06-25  73.74
9920  2026-06-26  70.16
9921  2026-06-29  71.59
             Date  Price
10186  2026-06-23  74.62
10187  2026-06-24  71.42
10188  2026-06-25  72.67
10189  2026-06-26  70.30
10190  2026-06-29  71.87


In [6]:
print(brent_daily.info())
print(brent_daily.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9922 entries, 0 to 9921
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    9922 non-null   object 
 1   Price   9922 non-null   float64
dtypes: float64(1), object(1)
memory usage: 155.2+ KB
None
             Price
count  9922.000000
mean     51.276754
std      32.772480
min       9.100000
25%      19.580000
50%      46.300000
75%      74.887500
max     143.950000


## Standardise the Data

In [7]:
import pandas as pd

def standardise_oil_data(df, date_col, price_col):

    df = df.copy()

    # Convert date column to datetime
    df[date_col] = pd.to_datetime(df[date_col])

    # Keep only date and price
    df = df[[date_col, price_col]]

    # Rename columns
    df.columns = ["Date", "Price"]

    # Sort by date
    df = df.sort_values("Date")

    # Remove duplicate rows
    df = df.drop_duplicates()

    # Remove missing values
    df = df.dropna()

    # Set Date as index
    df = df.set_index("Date")

    return df

In [8]:
oil_datasets = {
    "Brent Daily": standardise_oil_data(
        brent_daily,
        date_col="Date",
        price_col="Price"
    ),

    "Brent Weekly": standardise_oil_data(
        brent_weekly,
        date_col="Date",
        price_col="Price"
    ),

    "Brent Monthly": standardise_oil_data(
        brent_monthly,
        date_col="Date",
        price_col="Price"
    ),

    "Brent Yearly": standardise_oil_data(
        brent_yearly,
        date_col="Date",
        price_col="Price"
    ),

    "WTI Daily": standardise_oil_data(
        wti_daily,
        date_col="Date",
        price_col="Price"
    ),

    "WTI Weekly": standardise_oil_data(
        wti_weekly,
        date_col="Date",
        price_col="Price"
    ),

    "WTI Monthly": standardise_oil_data(
        wti_monthly,
        date_col="Date",
        price_col="Price"
    ),

    "WTI Yearly": standardise_oil_data(
        wti_yearly,
        date_col="Date",
        price_col="Price"
    )
}

In [9]:
brent_daily_clean = oil_datasets["Brent Daily"]
brent_weekly_clean = oil_datasets["Brent Weekly"]
brent_monthly_clean = oil_datasets["Brent Monthly"]
brent_yearly_clean = oil_datasets["Brent Yearly"]

wti_daily_clean = oil_datasets["WTI Daily"]
wti_weekly_clean = oil_datasets["WTI Weekly"]
wti_monthly_clean = oil_datasets["WTI Monthly"]
wti_yearly_clean = oil_datasets["WTI Yearly"]

In [10]:
for name, df in {
    "Brent Daily": brent_daily,
    "Brent Weekly": brent_weekly,
    "Brent Monthly": brent_monthly,
    "Brent Yearly": brent_yearly,
    "WTI Daily": wti_daily,
    "WTI Weekly": wti_weekly,
    "WTI Monthly": wti_monthly,
    "WTI Yearly": wti_yearly
}.items():

    print(name)
    print(df.columns.tolist())
    print()

Brent Daily
['Date', 'Price']

Brent Weekly
['Date', 'Price']

Brent Monthly
['Date', 'Price']

Brent Yearly
['Date', 'Price']

WTI Daily
['Date', 'Price']

WTI Weekly
['Date', 'Price']

WTI Monthly
['Date', 'Price']

WTI Yearly
['Date', 'Price']



In [11]:
print(brent_daily_clean.head())

            Price
Date             
1987-05-20  18.63
1987-05-21  18.45
1987-05-22  18.55
1987-05-25  18.60
1987-05-26  18.63


In [12]:
import os

# Create a folder for cleaned datasets
os.makedirs("cleaned_datasets", exist_ok=True)

# Save cleaned datasets
brent_daily_clean.to_csv("cleaned_datasets/brent_daily_clean.csv", index=True)
brent_weekly_clean.to_csv("cleaned_datasets/brent_weekly_clean.csv", index=True)
brent_monthly_clean.to_csv("cleaned_datasets/brent_monthly_clean.csv", index=True)
brent_yearly_clean.to_csv("cleaned_datasets/brent_yearly_clean.csv", index=True)

wti_daily_clean.to_csv("cleaned_datasets/wti_daily_clean.csv", index=True)
wti_weekly_clean.to_csv("cleaned_datasets/wti_weekly_clean.csv", index=True)
wti_monthly_clean.to_csv("cleaned_datasets/wti_monthly_clean.csv", index=True)
wti_yearly_clean.to_csv("cleaned_datasets/wti_yearly_clean.csv", index=True)